# Project Micro-TERRA — Colab A100 Distillation & Linux `io_uring` Benchmarks

Runs **Phase 2** of Micro-TERRA (`ANTIGRAVITY.md` Milestones 3B & 5) on Google Colab A100:
1. **Hardware & Drive Mount:** Auto-detects `A100-40GB` (4-bit NF4 frozen teacher + `BF16` student `BitLinear` + `PagedAdamW8bit`, ~32–35 GB VRAM) vs `A100-80GB` (`>= 75 GB` -> `BF16` teacher), checks `/tmp` NVMe space & Linux `io_uring` availability, and mounts `/content/drive/MyDrive/terra_ckpt`.
2. **Rust + Python Build:** Installs Python ML deps + `rustup` and compiles `edge_codec/` and `runtime/` (`AVX2`/`AVX-512` + `io_uring` `O_DIRECT`) with `RUSTFLAGS="-D warnings"`.
3. **Pre-Flight Parity Gates:** Runs `pytest` and `cargo test --release`.
4. **Auto-Resuming 7:1 Hybrid + 1.58-Bit Distillation:** Distills `allenai/OLMoE-1B-7B-0125` with combined loss `L_kl + 0.5 * L_layer_mse + 0.2 * L_lookahead_ce + 0.01 * L_load_balance` and atomic Drive checkpoints every 250 steps.
5. **Quality Gates:** Verifies `>= 80%` Lookahead Top-8 Recall ($L \to L+3$), WikiText-2 Perplexity retention (`>= 90%`), and HumanEval `pass@1` retention (`>= 90%`).
6. **Linux `io_uring` (`O_DIRECT`) NVMe Benchmarks & 16k Needle-in-a-Haystack:** Benchmarks Cold Demand vs. 3-Layer Lookahead under `--ram-budget-mb 2048` and persists final `core.bin`, `experts.bin`, `bench_results.json`, and `WALKTHROUGH.md` to Google Drive.

In [ ]:
import base64, io, os, pathlib, tarfile

_TERRA_SRC_TAR_XZ_B64 = "..."
_target_repo = pathlib.Path("/content/terra-llm")
if not (pathlib.Path.cwd() / "train" / "distill_colab.py").is_file() and not (_target_repo / "train" / "distill_colab.py").is_file():
    _target_repo.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_TERRA_SRC_TAR_XZ_B64)), mode="r:xz") as _tf:
        _tf.extractall(_target_repo)
    os.chdir(_target_repo)
    print(f"Extracted Micro-TERRA source bundle ({len(_TERRA_SRC_TAR_XZ_B64)} b64 chars) into {_target_repo}")

import os
import pathlib
import shutil
import subprocess
import sys
import torch

print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
!nvidia-smi
!df -h /tmp

io_uring_disabled_path = pathlib.Path("/proc/sys/kernel/io_uring_disabled")
if io_uring_disabled_path.exists():
    print("/proc/sys/kernel/io_uring_disabled =", io_uring_disabled_path.read_text().strip())
else:
    print("/proc/sys/kernel/io_uring_disabled not present (standard permissive kernel)")

# Probe O_DIRECT support on /tmp
probe_file = "/tmp/.terra_odirect_probe"
try:
    fd = os.open(probe_file, os.O_CREAT | os.O_RDWR | os.O_DIRECT, 0o644)
    os.close(fd)
    print("/tmp supports O_DIRECT: True")
except OSError as exc:
    print(f"/tmp O_DIRECT probe failed ({exc}); runtime pread fallback will be used automatically.")
finally:
    if os.path.exists(probe_file):
        os.remove(probe_file)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024.0 ** 3)
    mode = "4-bit NF4 Teacher + BF16 Student BitLinear (40GB mode)" if vram_gb < 75.0 else "BF16 Teacher (80GB mode)"
    print(f"GPU: {props.name} | VRAM: {vram_gb:.2f} GB -> {mode}")
else:
    print("WARNING: No CUDA GPU detected. Switch Colab Runtime -> Change runtime type -> A100 GPU.")

# Mount Google Drive for persistent atomic checkpoints across Colab timeouts
CKPT_DIR = pathlib.Path("/content/drive/MyDrive/terra_ckpt")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    CKPT_DIR = pathlib.Path("./terra_ckpt_local")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory:", CKPT_DIR)

# Locate repository root
for candidate in [pathlib.Path.cwd(), pathlib.Path("/content/terra-llm"), pathlib.Path("/content/drive/MyDrive/terra-llm")]:
    if (candidate / "train" / "distill_colab.py").is_file() and (candidate / "runtime" / "Cargo.toml").is_file():
        os.chdir(candidate)
        break
assert pathlib.Path("train/distill_colab.py").is_file(), "Upload or clone `terra-llm` into /content/terra-llm first!"
print("Active workspace:", pathlib.Path.cwd())

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "bitsandbytes>=0.43.0",
        "transformers>=4.46.0",
        "datasets>=2.20.0",
        "accelerate>=0.34.0",
        "safetensors>=0.4.5",
        "onnxruntime>=1.19.0",
        "psutil>=6.0.0",
        "human-eval",
        "pytest",
        "numpy",
        "huggingface_hub",
    ],
    check=True,
)

cargo_bin = pathlib.Path.home() / ".cargo" / "bin"
if str(cargo_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"

if shutil.which("cargo") is None:
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable",
        shell=True,
        check=True,
    )
    os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"

!rustc --version && cargo --version

env_rust = os.environ.copy()
env_rust["RUSTFLAGS"] = "-D warnings"
subprocess.run(["cargo", "build", "--release"], cwd="edge_codec", env=env_rust, check=True)
subprocess.run(["cargo", "build", "--release"], cwd="runtime", env=env_rust, check=True)
print("Built edge_codec and runtime --release binaries cleanly.")

In [ ]:
import os
import subprocess
import sys

env_rust = os.environ.copy()
env_rust["RUSTFLAGS"] = "-D warnings"

subprocess.run(["cargo", "test", "--release"], cwd="edge_codec", env=env_rust, check=True)
subprocess.run(["cargo", "test", "--release"], cwd="runtime", env=env_rust, check=True)
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "exporter", "edge_codec", "train/tests"],
    check=True,
)
print("All pre-flight Rust & Python parity tests passed on Linux!")

In [ ]:
import pathlib
import subprocess
import sys

# Ensure CKPT_DIR exists whether running interactively or top-to-bottom
ckpt_dir = pathlib.Path("/content/drive/MyDrive/terra_ckpt")
if not pathlib.Path("/content/drive").exists():
    ckpt_dir = pathlib.Path("./terra_ckpt_local")
ckpt_dir.mkdir(parents=True, exist_ok=True)

export_dir = pathlib.Path("/tmp/terra_olmoe")
export_dir.mkdir(parents=True, exist_ok=True)

# Background-friendly auto-resuming distillation:
#   - Saves atomic checkpoints every 250 steps (auto-resumes via `latest.json`)
#   - Exports `core.bin` + `experts.bin` to `/tmp/terra_olmoe` upon completion
cmd = [
    sys.executable,
    "-u",
    "-m",
    "train.distill_colab",
    "--teacher-model", "allenai/OLMoE-1B-7B-0125",
    "--ckpt-dir", str(ckpt_dir),
    "--export-dir", str(export_dir),
    "--steps", "3000",
    "--save-every", "250",
    "--seq-len", "1024",
    "--batch-size", "2",
    "--grad-accum", "8",
    "--lr", "2e-4",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pathlib
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from human_eval.data import read_problems
from human_eval.execution import check_correctness

from edge_codec import UnifiedEdgeCodec
from exporter.export_olmoe import run_reference_forward
from train.distill import build_student_from_teacher, evaluate_perplexity
from train.distill_colab import (
    collect_packed_eval_batches,
    detect_gpu_profile,
    evaluate_lookahead_recall,
    load_frozen_teacher,
    maybe_resume_latest,
)

MODEL_ID = "allenai/OLMoE-1B-7B-0125"
ckpt_dir = pathlib.Path("/content/drive/MyDrive/terra_ckpt")
if not pathlib.Path("/content/drive").exists():
    ckpt_dir = pathlib.Path("./terra_ckpt_local")
export_dir = pathlib.Path("/tmp/terra_olmoe")

gpu_profile = detect_gpu_profile()
device = torch.device(gpu_profile["backend"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 1. Load Student from latest checkpoint + Frozen Teacher
cpu_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, low_cpu_mem_usage=True)
student, lookahead_bank, _ = build_student_from_teacher(
    cpu_base,
    use_rms_prescale=False,
    init_l2_scale=False,
    enable_hint_embedding=True,
    deepcopy_model=False,
)
del cpu_base
opt_dummy = torch.optim.SGD(student.parameters(), lr=1e-4)
resumed_step = maybe_resume_latest(ckpt_dir, student, lookahead_bank, opt_dummy, None, map_location="cpu")
del opt_dummy
assert resumed_step > 0, f"No checkpoint found in {ckpt_dir}"

student = student.to(device=device, dtype=torch.bfloat16 if device.type == "cuda" else torch.float32)
student.eval()
teacher = load_frozen_teacher(MODEL_ID, gpu_profile)

# 2. Gate 1: Lookahead L -> L+3 Top-8 Recall (>= 80%)
val_batches = collect_packed_eval_batches(tokenizer, num_batches=16, seq_len=512, batch_size=2)
lookahead_recall = evaluate_lookahead_recall(student, lookahead_bank, val_batches, device)

# 3. Gate 2: WikiText-2 Perplexity (>= 90% retention)
wt2 = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wt2_text = "\n\n".join(x["text"] for x in wt2 if x["text"].strip())
wt2_ids = tokenizer.encode(wt2_text, add_special_tokens=False)[: 32 * 512]
wt2_batches = [
    torch.tensor(wt2_ids[i : i + 512], dtype=torch.long).unsqueeze(0)
    for i in range(0, len(wt2_ids) - 512 + 1, 512)
]
teacher_ppl = evaluate_perplexity(teacher, wt2_batches, device=device)
student_ppl = evaluate_perplexity(student, wt2_batches, device=device)
ppl_retention_pct = 100.0 * (teacher_ppl / max(student_ppl, 1e-6))

# 4. Gate 3: HumanEval pass@1 (>= 90% retention vs Teacher)
codec = UnifiedEdgeCodec(max_code_tokens=2048)
problems = read_problems()
def eval_humaneval_greedy(model_obj, is_student: bool, max_problems: int = 164) -> float:
    passed = 0
    items = list(problems.items())[:max_problems]
    for _, prob in items:
        prompt = prob["prompt"]
        if getattr(tokenizer, "is_fast", False):
            enc = tokenizer(prompt, return_offsets_mapping=True, return_tensors="pt")
            ids = enc["input_ids"].to(device)
            hints_t = None
            if is_student:
                raw_h = codec.encode_aligned_to_bpe(prompt, enc["offset_mapping"][0].tolist(), is_code=True)
                hints_t = torch.from_numpy(raw_h).unsqueeze(0).to(device)
        else:
            enc = tokenizer(prompt, return_tensors="pt").to(device)
            ids = enc["input_ids"]
            hints_t = None
            if is_student:
                raw_h = codec.encode_code(prompt, lang="python")
                h_pad = torch.zeros((1, ids.shape[1], 4), dtype=torch.int32, device=device)
                n_copy = min(h_pad.shape[1], raw_h.shape[0])
                h_pad[0, :n_copy] = torch.from_numpy(raw_h[:n_copy]).to(device)
                hints_t = h_pad
        with torch.no_grad():
            prompt_len = ids.shape[1]
            for _ in range(192):
                out = model_obj(input_ids=ids, hints=hints_t, use_cache=False) if is_student else model_obj(input_ids=ids, use_cache=False)
                next_id = int(out.logits[0, -1].argmax(dim=-1).item())
                ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
                if hints_t is not None:
                    hints_t = torch.cat([hints_t, torch.zeros((1, 1, 4), dtype=torch.int32, device=device)], dim=1)
                if next_id == tokenizer.eos_token_id:
                    break
        completion = tokenizer.decode(ids[0, prompt_len:], skip_special_tokens=True)
        for stop in ("\ndef ", "\nclass ", "\nif __name__"):
            if stop in completion:
                completion = completion.split(stop, 1)[0]
        res = check_correctness(prob, completion, timeout=3.0)
        if res["passed"]:
            passed += 1
    return passed / max(len(items), 1)

teacher_pass1 = eval_humaneval_greedy(teacher, is_student=False)
student_pass1 = eval_humaneval_greedy(student, is_student=True)
pass1_retention_pct = 100.0 * (student_pass1 / max(teacher_pass1, 1e-6))

# Write one prompt golden reference for `terra-cli --verify-golden`
if (export_dir / "core.bin").is_file():
    golden = run_reference_forward(export_dir, wt2_ids[:8])
    (export_dir / "golden_ref.json").write_text(json.dumps(golden, indent=2))

eval_summary = {
    "checkpoint_step": resumed_step,
    "lookahead_top8_recall": round(lookahead_recall, 4),
    "teacher_wikitext2_ppl": round(teacher_ppl, 3),
    "student_wikitext2_ppl": round(student_ppl, 3),
    "wikitext2_ppl_retention_pct": round(ppl_retention_pct, 2),
    "teacher_humaneval_pass1": round(teacher_pass1, 4),
    "student_humaneval_pass1": round(student_pass1, 4),
    "humaneval_retention_pct": round(pass1_retention_pct, 2),
}
(export_dir / "eval_gates.json").write_text(json.dumps(eval_summary, indent=2))
print(json.dumps(eval_summary, indent=2))

# Explicitly enforce Milestone 3B verification gates
assert lookahead_recall >= 0.80, f"Lookahead Top-8 recall {lookahead_recall:.2%} < 80%"
assert ppl_retention_pct >= 90.0, f"WikiText-2 PPL retention {ppl_retention_pct:.1f}% < 90%"
assert pass1_retention_pct >= 90.0, f"HumanEval pass@1 retention {pass1_retention_pct:.1f}% < 90%"

In [ ]:
import json
import pathlib
import subprocess
import sys
import torch
from transformers import AutoTokenizer

export_dir = pathlib.Path("/tmp/terra_olmoe")
terra_cli = pathlib.Path("./runtime/target/release/terra-cli")

# 1. Rust/PyTorch Golden Parity Check under strict --ram-budget-mb 2048
if (export_dir / "golden_ref.json").is_file():
    subprocess.run(
        [
            str(terra_cli),
            "--model-dir", str(export_dir),
            "--ram-budget-mb", "2048",
            "--verify-golden",
            "--golden-tol", "1e-2",
        ],
        check=True,
    )

# 2. Native Linux io_uring (O_DIRECT) vs Cold Demand NVMe Benchmark (--ram-budget-mb 2048 + 16k stream)
bench_out = export_dir / "bench_results.json"
subprocess.run(
    [
        sys.executable,
        "bench/run_benchmarks.py",
        "--model-dir", str(export_dir),
        "--ram-budget-mb", "2048",
        "--max-seq-len", "16384",
        "--run-16k-check",
        "--out-json", str(bench_out),
    ],
    check=True,
)
bench_payload = json.loads(bench_out.read_text())
for run in bench_payload["runs"]:
    r = run["report"]
    print(
        f"{run['label']}: io={r['io_backend']} direct={r['direct_io_active']} "
        f"hit={r['cache_hit_rate_pct']:.2f}% decode={r['decode_tokens_per_sec']:.2f} tok/s "
        f"stall={r['prefetch_metrics']['prefetch_stall_us']} us"
    )

# 3. 16k-Token Needle-in-a-Haystack Recall Probe (7:1 Hybrid Gated Recurrence + GQA Anchors)
# Uses `student` & `tokenizer` from the evaluation cell when present, plus exercises `terra-cli` at 16k context.
if "student" in globals() and "tokenizer" in globals():
    passkey = "7391042"
    filler_ids = tokenizer.encode("The coastal fog drifted past the granite lighthouse at dawn. ", add_special_tokens=False)
    needle_ids = tokenizer.encode(f"Important archival note: the secret launch passkey is {passkey}. ", add_special_tokens=False)
    query_ids = tokenizer.encode("Question: What is the secret launch passkey? Answer: ", add_special_tokens=False)

    pre_len = 4096
    post_len = 16000 - pre_len - len(needle_ids) - len(query_ids)
    pre_hay = (filler_ids * ((pre_len // len(filler_ids)) + 1))[:pre_len]
    post_hay = (filler_ids * ((post_len // len(filler_ids)) + 1))[:post_len]
    niah_ids = pre_hay + needle_ids + post_hay + query_ids
    assert len(niah_ids) == 16000

    inp = torch.tensor([niah_ids], dtype=torch.long, device=next(student.parameters()).device)
    with torch.no_grad():
        generated = []
        for _ in range(16):
            out = student(input_ids=inp, use_cache=False)
            nxt = int(out.logits[0, -1].argmax().item())
            generated.append(nxt)
            inp = torch.cat([inp, torch.tensor([[nxt]], dtype=torch.long, device=inp.device)], dim=1)
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    niah_pass = passkey in answer
    print(f"16k Needle-in-a-Haystack output: {answer!r} | Exact passkey recall: {niah_pass}")
    assert niah_pass, f"16k Needle-in-a-Haystack failed to recall {passkey}: {answer!r}"

In [ ]:
import pathlib
import shutil

export_dir = pathlib.Path("/tmp/terra_olmoe")
release_dir = pathlib.Path("/content/drive/MyDrive/terra_ckpt/final_release")
if not pathlib.Path("/content/drive").exists():
    release_dir = pathlib.Path("./terra_ckpt_local/final_release")
release_dir.mkdir(parents=True, exist_ok=True)

artifacts = [
    export_dir / "core.bin",
    export_dir / "experts.bin",
    export_dir / "bench_results.json",
    export_dir / "eval_gates.json",
    export_dir / "golden_ref.json",
]
for src in artifacts:
    if src.is_file():
        dst = release_dir / src.name
        shutil.copy2(src, dst)
        print(f"Archived {src.name} ({src.stat().st_size:,} bytes) -> {dst}")

for doc_candidate in [pathlib.Path("WALKTHROUGH.md"), pathlib.Path("walkthrough.md")]:
    if doc_candidate.is_file():
        shutil.copy2(doc_candidate, release_dir / "WALKTHROUGH.md")
        print(f"Archived {doc_candidate} -> {release_dir / 'WALKTHROUGH.md'}")
        break